In [1]:
import open3d as o3d
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import glob


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
pcd_files = glob.glob("/home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/*.pcd")

print(len(pcd_files))

158


In [4]:
def clean_cloud(pcd):
    pcd, _ = pcd.remove_statistical_outlier(
        nb_neighbors=20,
        std_ratio=2.0
    )
    return pcd


In [5]:
def extract_plane(pcd):

    plane_model, inliers = pcd.segment_plane(
        distance_threshold=0.02,
        ransac_n=3,
        num_iterations=1000
    )

    plane = pcd.select_by_index(inliers)
    rest = pcd.select_by_index(inliers, invert=True)

    return plane, rest


In [6]:
def cluster_objects(pcd):

    labels = np.array(
        pcd.cluster_dbscan(eps=0.05, min_points=50)
    )

    clusters = []

    for i in range(labels.max() + 1):
        idx = np.where(labels == i)[0]
        cluster = pcd.select_by_index(idx)
        clusters.append(cluster)

    return clusters


In [7]:
def extract_features(cluster):

    points = np.asarray(cluster.points)

    if len(points) < 10:
        return None

    cov = np.cov(points.T)
    eigenvalues = np.linalg.eigvals(cov)

    bbox = cluster.get_axis_aligned_bounding_box()

    return [
        len(points),
        bbox.volume(),
        eigenvalues[0],
        eigenvalues[1],
        eigenvalues[2]
    ]


In [8]:
dataset = []

for file in pcd_files:

    print("Procesando:", file)

    pcd = o3d.io.read_point_cloud(file)
    pcd = clean_cloud(pcd)

    plane, rest = extract_plane(pcd)

    # Etiqueta automática
    dataset.append(extract_features(plane) + ["wall"])

    clusters = cluster_objects(rest)

    for cluster in clusters:

        o3d.visualization.draw_geometries([cluster])

        label = input("Etiqueta cluster (rubble/unknown/skip): ")

        if label == "skip":
            continue

        features = extract_features(cluster)

        if features:
            dataset.append(features + [label])


Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208503.775408268.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208496.275605202.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208497.875730276.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208502.976123333.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208500.175746202.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208493.376474380.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208492.976199389.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208507.175581217.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208505.176241159.pcd
Procesando: /home/labia-001/Repo/turi-project/docs/Pruebas_posicion_2/1758208492.275396347.pcd
Procesando: /home/labia-001/Repo/turi-project/docs

In [9]:
df = pd.DataFrame(dataset,
                  columns=[
                      "num_points",
                      "volume",
                      "eig1",
                      "eig2",
                      "eig3",
                      "label"
                  ])

df.to_csv("lidar_dataset.csv", index=False)


In [10]:
from sklearn.model_selection import train_test_split

X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))


Accuracy: 1.0


In [11]:
import joblib
joblib.dump(model, "semantic_model.pkl")


['semantic_model.pkl']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d

def visualizar_clusters(pcd):

    labels = np.array(
        pcd.cluster_dbscan(eps=0.05, min_points=50)
    )

    max_label = labels.max()
    print(f"Clusters encontrados: {max_label + 1}")

    colors = plt.get_cmap("tab20")(
        labels / (max_label if max_label > 0 else 1)
    )

    colors[labels < 0] = 0
    pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])

    o3d.visualization.draw_geometries([pcd])

In [14]:
pcd = o3d.io.read_point_cloud(pcd_files[0])
visualizar_clusters(pcd)


Clusters encontrados: 0
